# Встановимо API ключ для OpenAI

In [4]:
import os

os.environ["OPENAI_API_KEY"] = input("Введіть дійсний OpenAI API ключ")

# Визначимо функцію побудови агента технічної підтримки

In [6]:
from functools import cache
from langgraph.constants import START, END
from langgraph.types import RetryPolicy
from lib.nodes import classify_query, search_faq, draft_response, check_escalation
from lib.classification import SupportAgentState
from langgraph.graph import StateGraph
from langgraph.graph.state import CompiledStateGraph

@cache
def build_support_agent() -> CompiledStateGraph:
    """Build a support agent."""
    workflow = StateGraph(SupportAgentState)
    workflow.add_node("classify_query", classify_query)
    workflow.add_node("search_faq", search_faq, retry_policy=RetryPolicy(max_attempts=3))
    workflow.add_node("draft_response", draft_response)
    workflow.add_node("check_escalation", check_escalation)

    workflow.add_edge(START, "classify_query")
    workflow.add_edge("classify_query", "search_faq")
    workflow.add_edge("search_faq", "draft_response")
    workflow.add_edge("draft_response", "check_escalation")
    workflow.add_edge("check_escalation", END)

    return workflow.compile()

# А тепер все перетестуємо

In [7]:
from pydantic import SecretStr
from langchain_openai import ChatOpenAI

# Ініціалізуємо LLM
api_key = os.environ.get("OPENAI_API_KEY")
llm = ChatOpenAI(model="gpt-4", temperature=0.1, api_key=SecretStr(api_key))

# Оцінка релевантності
def calculate_answer_relevancy(query: str, response: str) -> float:
    """Оцінка релевантності"""

    relevancy_prompt = f"""
    Оцінка релевантності (0-1):
    Запит: {query}
    Відповідь: {response}

    Тільки число:
    """

    try:
        result = llm.invoke(relevancy_prompt)
        text = result.content.strip()
        score = float(''.join(c for c in text if c.isdigit() or c == '.'))
        return min(max(score, 0.0), 1.0)
    except:
        return 0.5

def test_agent():
    """Тестування агента на 5 запитах"""

    agent = build_support_agent()

    test_queries = [
        "Я забув пароль від свого акаунту, як його відновити?",
        "З мене двічі зняли гроші за підписку! Це шахрайство! Поверніть кошти негайно!",
        "Чи можу я змінити тарифний план на дешевший?",
        "Сайт не працює вже третю годину! Коли полагодите?",
        "Як додати нового користувача до корпоративного акаунту?"
    ]

    results = []
    relevancy_scores = []

    print("=" * 70)
    print("ТЕСТУВАННЯ АГЕНТА ТЕХНІЧНОЇ ПІДТРИМКИ")
    print("=" * 70)

    for i, query in enumerate(test_queries, 1):
        print(f"\\n{'='*70}")
        print(f"📧 ТЕСТ {i}/5")
        print(f"{'='*70}")
        print(f"ЗАПИТ: {query}")
        print("-" * 70)

        initial_state = {
            "user_query": query,
            "needs_escalation": False
        }

        config = {"configurable": {"thread_id": f"test_{i}"}}

        try:
            result = agent.invoke(initial_state, config)
            results.append(result)

            # Виводимо класифікацію
            if result.get("classification"):
                cls = result["classification"]
                print(f"\\n📊 КЛАСИФІКАЦІЯ:")
                print(f"   Тип: {cls.get('type')}")
                print(f"   Категорія: {cls.get('category')}")
                print(f"   Терміновість: {cls.get('urgency')}")
                print(f"   Резюме: {cls.get('summary')}")

            # Виводимо знайдені FAQ
            if result.get("search_results"):
                print(f"\\n🔍 FAQ ({len(result['search_results'])} пунктів):")
                for j, faq in enumerate(result['search_results'], 1):
                    print(f"   {j}. {faq}")

            # ВИВОДИМО ВІДПОВІДЬ АГЕНТА
            if result.get("draft_response"):
                print(f"\\n💬 ВІДПОВІДЬ АГЕНТА:")
                print("-" * 70)
                print(result["draft_response"])
                print("-" * 70)

                # Рахуємо релевантність
                relevancy = calculate_answer_relevancy(query, result["draft_response"])
                relevancy_scores.append(relevancy)
                print(f"\\n📈 Релевантність: {relevancy:.0%}")

            # Статус ескалації
            if result.get("needs_escalation"):
                print("\\n⚠️ СТАТУС: Ескаловано до спеціаліста")
            else:
                print("\\n✅ СТАТУС: Оброблено автоматично")

        except Exception as e:
            print(f"❌ ПОМИЛКА: {e}")
            results.append({})

    # Фінальні метрики
    print(f"\\n{'='*70}")
    print("ПІДСУМКОВІ МЕТРИКИ")
    print("=" * 70)

    # Task Completion Rate
    completed = sum(1 for r in results if r.get("draft_response"))
    tcr = completed / len(results) if results else 0
    print(f"✅ Task Completion Rate: {tcr:.0%} ({completed}/{len(results)})")

    # Answer Relevancy
    if relevancy_scores:
        avg_relevancy = sum(relevancy_scores) / len(relevancy_scores)
        print(f"🎯 Average Answer Relevancy: {avg_relevancy:.0%}")

    # Escalation Rate
    escalated = sum(1 for r in results if r.get("needs_escalation"))
    print(f"📊 Escalation Rate: {escalated}/{len(results)} ({escalated/len(results)*100:.0f}%)")

    print("=" * 70)

    return results

print(test_agent())

ТЕСТУВАННЯ АГЕНТА ТЕХНІЧНОЇ ПІДТРИМКИ
\n======================================================================
📧 ТЕСТ 1/5
ЗАПИТ: Я забув пароль від свого акаунту, як його відновити?
----------------------------------------------------------------------
\n📊 КЛАСИФІКАЦІЯ:
   Тип: problem
   Категорія: password
   Терміновість: medium
   Резюме: User forgot their account password and needs help to reset it.
\n💬 ВІДПОВІДЬ АГЕНТА:
----------------------------------------------------------------------
Вітаємо! Щоб відновити пароль від вашого акаунту, будь ласка, виконайте наступні кроки:

1. Натисніть на посилання 'Забули пароль?' на сторінці входу.
2. Введіть вашу електронну адресу, пов'язану з акаунтом.
3. Перевірте вашу поштову скриньку для отримання листа з інструкціями щодо скидання пароля.

Якщо ви не отримали листа, будь ласка, перевірте папку 'Спам'.

Новий пароль повинен містити мінімум 8 символів, включаючи літери та цифри. Якщо у вас виникнуть додаткові питання, не соромтеся зверт

# Ставте свої запитання

In [10]:
initial_state = {
            "user_query": input("Аліса: Чим можу допомогти?"),
            "needs_escalation": False
        }
agent = build_support_agent()
try:
    result = agent.invoke(initial_state)
    print("Запит: ", initial_state["user_query"])
    print("Відповідь: ", result["draft_response"])
    print("Ескальовано до технічного спеціаліста" if result["needs_escalation"] else "Не потребує ескалації")
except Exception as e:
    print(f"❌ ПОМИЛКА: {e}")

Запит:  При другому звернені до дашборду мого аккаунта, я помітив, що всі кошти просто зникли. Що мені робити?
Відповідь:  Дякуємо за ваше звернення. Ми розуміємо, що ситуація зникнення коштів з вашого аккаунта може бути дуже тривожною. Ось кілька кроків, які ви можете спробувати:

1. **Перевірте правильність введеного email**: Іноді проблема може бути пов'язана з неправильним email, що використовується для входу в аккаунт.

2. **Оновіть сторінку або спробуйте інший браузер**: Це може допомогти, якщо проблема пов'язана з технічними неполадками.

3. **Зверніться до нашої служби підтримки**: Якщо проблема не вирішується, будь ласка, зв'яжіться з нашою службою підтримки, яка працює 24/7, для подальшої допомоги.

Ми рекомендуємо негайно звернутися до підтримки, оскільки ваш запит має високий пріоритет. Вибачте за незручності, які це могло спричинити.
Ескальовано до технічного спеціаліста
